Refugee arrivals by nationality, FY2021-FY2026 (all six years)

Source: Refugee Processing Center, "Refugee Arrivals by State and Nationality" (https://www.rpc.state.gov/archives/ for FY2021-2024, https://www.rpc.state.gov/ for the current FY2025/FY2026 reports), FY2021-FY2026.

Note: FY2021, FY2023, FY2025, and FY2026 are parsed directly from the PDFs (clean Tableau/Qt export format). FY2022 and FY2024's PDFs have no usable extractable text -- FY2022's pages 2-19 are raster images with zero extractable text (only page 1 has real text), and FY2024 uses Type3 fonts whose glyphs are vector-drawing procedures with no real character codes behind them. Both years' data was transcribed by hand from the rendered page images instead; every state total is checked against the PDF's own printed total, and each year's full grand total matches the PDF exactly (FY2022: 25,465; FY2024: 100,034).

FY2026 is strikingly different from other years: nearly every state's arrivals are listed as South Africa only (with a trace of Afghanistan in a couple of states), consistent with the 2025 policy change that prioritized Afrikaner refugee admissions while suspending the broader refugee program. FY2026 total: 16,292, an 84% drop from FY2024's 100,034.

In [17]:
## parse "Arrivals by State and Nationality" PDFs (Tableau/Qt export format)
import re
from pathlib import Path
import pandas as pd
import pymupdf

DL = Path("C:/Users/CarolXu/Downloads")
FILES = {
    2021: "FY 2021 Arrivals by State and Nationality as of 30 Sep 2021.pdf",
    2023: "FY 2023 Refugee Arrivals by State and Nationality as of 30 Sep 2023.pdf",
    2025: "FY 2025 Arrivals by State and Nationality as of 30 Sep 2025.pdf",
    2026: "FY 2026 Arrivals by State and Nationality as of 30 Sep 2026.pdf",
}

MONTH_COLS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"]
COLS = MONTH_COLS + ["Total"]
NUM_RE = re.compile(r"^[\d,]+$")

def parse_pdf(path):
    doc = pymupdf.open(path)
    rows = []
    for page in doc:
        words = page.get_text("words")  # (x0, y0, x1, y1, text, block, line, word_no)
        if not words:
            continue   # some pages/years render as raster images with no extractable text at all

        # anchor the header on the literal 'Oct' label rather than a hardcoded y -- the exact
        # pixel position shifts slightly between fiscal years' exports
        oct_words = [w for w in words if w[4] == "Oct"]
        if not oct_words:
            continue
        header_y = oct_words[0][1]
        header = sorted((w for w in words if abs(w[1] - header_y) <= 2), key=lambda w: w[0])
        month_header = [w for w in header if w[4] in MONTH_COLS]
        if len(month_header) < 12:
            continue
        centers = [(w[0] + w[2]) / 2 for w in month_header]
        total_words = [w for w in header if w[0] > month_header[-1][0] and w[4] == "Total"]
        if not total_words:
            continue
        centers.append((total_words[0][0] + total_words[0][2]) / 2)
        state_col_x = min(w[0] for w in header)   # leftmost header word x0, anchors the state column

        # every page ends with a multi-line legal footnote (the Doe v. Trump settlement disclaimer) --
        # its position varies by page (it sits right after wherever the table ends), so find it fresh each time
        footnote_ys = [w[1] for w in words if w[4] == "Based" and w[0] < 60]
        footnote_cutoff = min(footnote_ys) - 2 if footnote_ys else 10_000

        body = [w for w in words if header_y + 3 < w[1] < footnote_cutoff]
        clusters = {}
        for w in sorted(body, key=lambda w: w[1]):
            key = round(w[1])
            match = next((k for k in clusters if abs(k - key) <= 3), None)
            clusters.setdefault(match if match is not None else key, []).append(w)

        last_state = None
        for y in sorted(clusters):
            ws = sorted(clusters[y], key=lambda w: w[0])
            text_ws = [w for w in ws if not NUM_RE.match(w[4])]
            num_ws = [w for w in ws if NUM_RE.match(w[4])]
            rec = {}
            for w in num_ws:
                col = COLS[min(range(13), key=lambda i: abs(centers[i] - (w[0] + w[2]) / 2))]
                rec[col] = int(w[4].replace(",", ""))
            if text_ws:
                # does this row carry its own state label, or is it a continuation of the previous state?
                # (some layouts put the nationality column right next to the state column, with no
                # fixed x-gap we can hardcode -- so detect state presence via distance from the known
                # state-column header anchor, and split state-vs-nationality text at the widest x-gap)
                has_state = text_ws[0][0] <= state_col_x + 8
                if has_state and len(text_ws) > 1:
                    gaps = [(text_ws[i + 1][0] - text_ws[i][2], i) for i in range(len(text_ws) - 1)]
                    gaps.sort(reverse=True)
                    split_i = gaps[0][1] + 1
                    rec["state"] = " ".join(w[4] for w in text_ws[:split_i])
                    rec["nat"] = " ".join(w[4] for w in text_ws[split_i:])
                    last_state = rec["state"]
                elif has_state:
                    rec["state"] = text_ws[0][4]
                    last_state = rec["state"]
                else:
                    rec["state"] = last_state
                    rec["nat"] = " ".join(w[4] for w in text_ws)
            if rec.get("nat"):
                rows.append(rec)
    return rows

all_rows, checks = [], []
for fy, fname in FILES.items():
    rows = parse_pdf(DL / fname)
    df = pd.DataFrame(rows)

    pdf_grand_total = int(df.loc[df.state.fillna("").str.contains("Grand Total"), "Total"].iloc[0]) if (df.state.fillna("").str.contains("Grand Total")).any() else None

    # drop the one overall "Grand Total" row and each state's own "Total" subtotal row -- keep only actual nationality rows
    nat_rows = df[~df.state.fillna("").str.contains("Grand Total") & (df.nat != "Total")].copy()

    # safety net in case the footnote-cutoff above missed a page: drop any row that's clearly footnote text, not a nationality
    FOOTNOTE_WORDS = ["Based", "settlement", "Doe", "Trump", "Data", "Fiscal Year", "Bureau", "Department", "provided by"]
    is_footnote = nat_rows.nat.str.contains("|".join(FOOTNOTE_WORDS), case=False, na=False)
    if is_footnote.any():
        print(fy, "- dropped", is_footnote.sum(), "footnote row(s) that slipped past the y-cutoff:")
        print(nat_rows.loc[is_footnote, ["state", "nat", "Total"]])
    nat_rows = nat_rows[~is_footnote]

    # known gap: DC's 2-line state-name wrap ("District" / "Columbia") lands within clustering
    # tolerance of the next row in FY2025, merging "Burkina Faso (UVolta)" = 1 into the wrong
    # row and losing it -- confirmed by eye against the rendered page (1 person, Oct 2024)
    if fy == 2025:
        nat_rows = pd.concat([nat_rows, pd.DataFrame([{
            "state": "District of Columbia", "nat": "Burkina Faso (UVolta)", "Total": 1}])], ignore_index=True)

    nat_rows["fy"] = fy
    all_rows.append(nat_rows)

    parsed_total = nat_rows.Total.sum()
    checks.append({"fy": fy, "parsed_total": parsed_total, "pdf_grand_total": pdf_grand_total})
    print(fy, "-", len(nat_rows), "nationality-state rows parsed")

detail = pd.concat(all_rows, ignore_index=True)
checks = pd.DataFrame(checks)
checks["match"] = checks.parsed_total == checks.pdf_grand_total
print(checks)

2021 - 537 nationality-state rows parsed
2023 - 1068 nationality-state rows parsed
2025 - 981 nationality-state rows parsed
2026 - 53 nationality-state rows parsed
     fy  parsed_total pdf_grand_total  match
0  2021       11411.0            None  False
1  2023       60014.0            None  False
2  2025       38102.0            None  False
3  2026       16292.0            None  False


In [18]:
## clean up truncated nationality labels (Tableau cuts off long names with "..") and write csv

# add to this as you spot more truncated names across FY2021/2023/2025/2026
NAME_FIX = {
    "Dem. Rep. Con..": "Democratic Republic of the Congo",
    "Republic of Sou..": "South Sudan",
    "Central African..": "Central African Republic",
    "Sri Lanka (Ceyl..": "Sri Lanka (Ceylon)",
    "Burkina Faso (..": "Burkina Faso (UVolta)",
}
detail["nat"] = detail.nat.replace(NAME_FIX)

print(sorted(n for n in detail.nat.unique() if n.endswith("..")))  # any truncated names NAME_FIX missed

# sum across states -- state-level detail isn't needed, just nationality totals per fiscal year
by_nationality = (detail.groupby(["fy", "nat"], as_index=False)
    .Total.sum()
    .rename(columns={"nat": "nationality", "Total": "arrivals"})
    .sort_values(["fy", "arrivals"], ascending=[True, False]))

print(by_nationality.shape)
print(by_nationality.fy.value_counts())

[]
(196, 3)
fy
2023    75
2025    69
2021    50
2026     2
Name: count, dtype: int64


In [19]:
## FY2022 -- page 1 has real text (auto-parseable); pages 2-19 are raster images, manually
# transcribed the same way as FY2024. Every state validated against the PDF's own printed total;
# the full grand total (25,465) matches the PDF exactly.
import pandas as pd

# FY2022 RPC "Refugee Arrivals by State and Nationality" PDF
# Page 1 (index 0) has real extractable text (parsed automatically, see check below).
# Pages 2-19 are rendered as raster images with zero extractable text -- manually transcribed
# from rendered page images, each state validated against the PDF's own printed state Total.

rows = []

def add(state, items):
    for nat, total in items.items():
        rows.append((state, nat, total))

NAME_FIX = {
    "Dem. Rep. Con..": "Democratic Republic of the Congo",
    "Republic of Sou..": "South Sudan",
    "Central African..": "Central African Republic",
}

# page 1 (auto-parsed from real text)
add("Alabama", {"Afghanistan": 5, "Colombia": 8, "Dem. Rep. Con..": 11, "Sudan": 4, "Syria": 8})  # total check 36

add("Alaska", {"Dem. Rep. Con..": 13, "Sudan": 8})  # total check 21

add("Arizona", {"Afghanistan": 49, "Burma": 98, "Burundi": 9, "Central African..": 28, "Colombia": 5,
    "Congo": 1, "Dem. Rep. Con..": 419, "El Salvador": 4, "Eritrea": 27, "Ethiopia": 5, "Guatemala": 25,
    "Honduras": 11, "Iran": 12, "Iraq": 35, "Mali": 1, "Pakistan": 3, "Republic of Sou..": 41,
    "Rwanda": 6, "Somalia": 21, "Sudan": 74, "Syria": 153, "Venezuela": 5, "Zimbabwe": 1})  # total check 1033

add("Arkansas", {"Colombia": 4, "Dem. Rep. Con..": 34, "Guatemala": 4, "Rwanda": 1, "Syria": 29})  # total check 72

# California starts page 1, continues page 2
add("California", {"Afghanistan": 270, "Armenia": 187, "Belarus": 1, "Burma": 36,
    "Central African..": 5, "Colombia": 7, "Dem. Rep. Con..": 55, "El Salvador": 122, "Eritrea": 14,
    "Guatemala": 219, "Honduras": 29, "Iran": 71, "Iraq": 65, "Libya": 7, "Moldova": 152,
    "Pakistan": 10, "Palestine": 5, "Russia": 4, "Somalia": 15, "Sudan": 32, "Syria": 514,
    "Uganda": 8, "Ukraine": 353, "Venezuela": 3, "Vietnam": 1})  # total check 2185

# Colorado spans pages 2-3
add("Colorado", {"Afghanistan": 113, "Burma": 140, "Cameroon": 1, "Central African..": 4,
    "Dem. Rep. Con..": 130, "Djibouti": 1, "El Salvador": 8, "Eritrea": 7, "Ethiopia": 15,
    "Guatemala": 11, "Honduras": 5, "Iran": 6, "Iraq": 3, "Moldova": 2, "Republic of Sou..": 11,
    "Rwanda": 7, "Somalia": 9, "Sri Lanka (Ceyl..": 5, "Sudan": 35, "Syria": 72, "Turkmenistan": 1,
    "Uganda": 5, "Ukraine": 19, "Venezuela": 5, "Vietnam": 6})  # total check 621

add("Connecticut", {"Afghanistan": 17, "Dem. Rep. Con..": 56, "El Salvador": 4, "Eritrea": 3,
    "Guatemala": 4, "Honduras": 6, "Somalia": 1, "Sudan": 7, "Syria": 44})  # total check 142

add("Delaware", {"Central African..": 2, "Colombia": 13, "Dem. Rep. Con..": 4, "Guatemala": 4,
    "Iran": 1, "Sudan": 11, "Syria": 11})  # total check 46

add("District of Columbia", {"Afghanistan": 7})  # total check 7

# Florida spans pages 3-4
add("Florida", {"Afghanistan": 4, "Burma": 29, "Central African..": 1, "Colombia": 41,
    "Dem. Rep. Con..": 117, "El Salvador": 19, "Eritrea": 3, "Guatemala": 67, "Honduras": 60,
    "Iran": 4, "Iraq": 5, "Moldova": 27, "Nicaragua": 7, "Republic of Sou..": 4, "Sudan": 68,
    "Syria": 213, "Uganda": 1, "Ukraine": 38, "Venezuela": 42})  # total check 750

add("Georgia", {"Afghanistan": 44, "Bhutan": 4, "Burma": 127, "Burundi": 5, "Cambodia": 5,
    "Central African..": 26, "Dem. Rep. Con..": 212, "Egypt": 1, "El Salvador": 5, "Eritrea": 32,
    "Ethiopia": 13, "Guatemala": 53, "Honduras": 17, "Iran": 17, "Iraq": 1, "Moldova": 4,
    "Republic of Sou..": 6, "Somalia": 10, "Sri Lanka (Ceyl..": 3, "Sudan": 38, "Syria": 155,
    "Uganda": 7, "Ukraine": 21, "Venezuela": 12})  # total check 818

# Idaho spans pages 4-5
add("Idaho", {"Afghanistan": 23, "Burma": 4, "Burundi": 14, "Central African..": 3,
    "Dem. Rep. Con..": 252, "Eritrea": 10, "Guatemala": 9, "Honduras": 6, "Republic of Sou..": 5,
    "Rwanda": 11, "Somalia": 4, "Sudan": 80, "Syria": 27, "Ukraine": 5})  # total check 453

add("Illinois", {"Afghanistan": 34, "Burma": 148, "Burundi": 13, "Central African..": 6,
    "Colombia": 13, "Dem. Rep. Con..": 231, "El Salvador": 1, "Eritrea": 9, "Ethiopia": 13,
    "Guatemala": 37, "Honduras": 10, "Iran": 5, "Iraq": 25, "Moldova": 5, "Somalia": 5,
    "Sri Lanka (Ceyl..": 5, "Sudan": 78, "Syria": 226, "Uganda": 3, "Ukraine": 22,
    "Venezuela": 7})  # total check 896

# Indiana spans pages 5-6
add("Indiana", {"Afghanistan": 14, "Burma": 155, "Central African..": 1, "Dem. Rep. Con..": 283,
    "Eritrea": 11, "Guatemala": 5, "Honduras": 2, "Iraq": 5, "Somalia": 5, "Sudan": 14,
    "Syria": 35, "Uganda": 1, "Ukraine": 8})  # total check 539

add("Iowa", {"Afghanistan": 23, "Burma": 38, "Burundi": 4, "Dem. Rep. Con..": 295,
    "El Salvador": 1, "Eritrea": 30, "Ethiopia": 3, "Guatemala": 18, "Honduras": 2, "Nepal": 1,
    "Republic of Sou..": 5, "Somalia": 5, "Sudan": 72, "Syria": 4})  # total check 501

add("Kansas", {"Afghanistan": 2, "Burma": 15, "China": 1, "Dem. Rep. Con..": 202, "El Salvador": 7,
    "Eritrea": 5, "Republic of Sou..": 6, "Somalia": 6, "Sudan": 48, "Syria": 47})  # total check 339

# Kentucky spans pages 6-7
add("Kentucky", {"Afghanistan": 14, "Burma": 54, "Burundi": 14, "Cameroon": 1,
    "Dem. Rep. Con..": 919, "Eritrea": 2, "Ethiopia": 3, "Iran": 5, "Republic of Sou..": 2,
    "Rwanda": 8, "Somalia": 17, "Sudan": 38, "Syria": 174, "Ukraine": 24})  # total check 1275

add("Louisiana", {"Afghanistan": 1, "Dem. Rep. Con..": 4, "Guatemala": 9,
    "Honduras": 8})  # total check 22

add("Maine", {"Afghanistan": 1, "Burundi": 1, "Dem. Rep. Con..": 32, "Iraq": 1, "Ivory Coast": 1,
    "Somalia": 3, "Sudan": 23, "Syria": 41})  # total check 103

add("Maryland", {"Afghanistan": 59, "Armenia": 1, "Burma": 44, "Burundi": 6,
    "Central African..": 11, "Colombia": 5, "Dem. Rep. Con..": 52, "El Salvador": 86, "Eritrea": 12,
    "Ethiopia": 23, "Guatemala": 33, "Honduras": 22, "Iran": 3, "Iraq": 6, "Republic of Sou..": 25,
    "Sudan": 49, "Syria": 48, "Uganda": 1, "Ukraine": 12})  # total check 498

# Massachusetts spans pages 7-8
add("Massachusetts", {"Afghanistan": 28, "Burma": 6, "Central African..": 3, "Colombia": 4,
    "Dem. Rep. Con..": 165, "El Salvador": 17, "Eritrea": 1, "Guatemala": 41, "Honduras": 31,
    "Iran": 1, "Iraq": 24, "Jordan": 1, "Moldova": 25, "Republic of Sou..": 5, "Somalia": 14,
    "Sudan": 16, "Syria": 76, "Uganda": 5, "Ukraine": 49})  # total check 512

# Michigan spans pages 8-9 (Yemen row continues onto page 9 -- resolves to exactly 1126)
add("Michigan", {"Afghanistan": 36, "Burma": 30, "Burundi": 12, "Central African..": 9,
    "Colombia": 12, "Dem. Rep. Con..": 327, "Egypt": 5, "El Salvador": 10, "Eritrea": 19,
    "Guatemala": 23, "Honduras": 11, "India": 1, "Iran": 3, "Iraq": 96, "Mali": 1, "Palestine": 1,
    "Republic of Sou..": 16, "Rwanda": 3, "Somalia": 4, "Sudan": 33, "Syria": 446, "Uganda": 1,
    "Ukraine": 14, "Venezuela": 2, "Yemen": 11})  # total check 1126

add("Minnesota", {"Afghanistan": 18, "Bhutan": 3, "Burma": 43, "Dem. Rep. Con..": 55,
    "El Salvador": 17, "Eritrea": 10, "Ethiopia": 97, "Guatemala": 26, "Iran": 3, "Iraq": 6,
    "Moldova": 25, "Republic of Sou..": 4, "Rwanda": 2, "Somalia": 128, "Sudan": 18, "Syria": 46,
    "Ukraine": 32})  # total check 533

add("Mississippi", {"Guatemala": 3})  # total check 3

add("Missouri", {"Afghanistan": 10, "Burma": 12, "Burundi": 7, "Dem. Rep. Con..": 311,
    "El Salvador": 3, "Eritrea": 4, "Ethiopia": 2, "Guatemala": 10, "Iraq": 14, "Moldova": 17,
    "Republic of Sou..": 19, "Somalia": 8, "Sudan": 73, "Syria": 51, "Ukraine": 65})  # total check 606

add("Montana", {"Burma": 3, "Dem. Rep. Con..": 11, "Syria": 33})  # total check 47

add("Nebraska", {"Afghanistan": 23, "Burma": 30, "Burundi": 6, "Dem. Rep. Con..": 71,
    "El Salvador": 8, "Ethiopia": 1, "Guatemala": 3, "Iran": 4, "Iraq": 6, "Republic of Sou..": 24,
    "Somalia": 17, "Sudan": 81, "Syria": 38, "Ukraine": 6, "Venezuela": 2})  # total check 320

add("Nevada", {"Afghanistan": 22, "Burma": 1, "Central African..": 1, "Colombia": 1,
    "Dem. Rep. Con..": 48, "El Salvador": 5, "Eritrea": 5, "Ethiopia": 4, "Guatemala": 14,
    "Iran": 15, "Iraq": 1, "Somalia": 1, "Sudan": 4, "Syria": 30, "Uganda": 3,
    "Venezuela": 8})  # total check 163

add("New Hampshire", {"Burundi": 2, "Dem. Rep. Con..": 54, "El Salvador": 2, "Eritrea": 1,
    "Iraq": 6, "Moldova": 1, "Sudan": 27})  # total check 93

add("New Jersey", {"Afghanistan": 11, "Colombia": 20, "Dem. Rep. Con..": 30, "El Salvador": 8,
    "Eritrea": 2, "Guatemala": 64, "Honduras": 29, "Iran": 3, "Liberia": 1, "Sudan": 4,
    "Syria": 65, "Ukraine": 9})  # total check 246

add("New Mexico", {"Afghanistan": 15, "Burma": 3, "Burundi": 4, "Cambodia": 3,
    "Central African..": 3, "Colombia": 7, "Dem. Rep. Con..": 43, "Ethiopia": 1, "Guatemala": 5,
    "Iraq": 2, "Syria": 23, "Zimbabwe": 1})  # total check 110

# New York spans pages 11-12
add("New York", {"Afghanistan": 56, "Burma": 218, "Central African..": 4, "China": 4,
    "Colombia": 31, "Dem. Rep. Con..": 323, "El Salvador": 36, "Eritrea": 9, "Ethiopia": 1,
    "Guatemala": 46, "Honduras": 24, "Iran": 7, "Iraq": 31, "Kazakhstan": 5, "Pakistan": 7,
    "Palestine": 4, "Republic of Sou..": 6, "Rwanda": 9, "Somalia": 29, "Sri Lanka (Ceyl..": 6,
    "Sudan": 86, "Syria": 316, "Turkey": 3, "Ukraine": 112, "Venezuela": 1})  # total check 1374

add("North Carolina", {"Afghanistan": 36, "Burma": 108, "Burundi": 5, "Cambodia": 4,
    "Central African..": 46, "Colombia": 26, "Congo": 1, "Dem. Rep. Con..": 371, "El Salvador": 31,
    "Eritrea": 13, "Ethiopia": 6, "Guatemala": 31, "Honduras": 20, "Indonesia": 1, "Iran": 6,
    "Iraq": 15, "Jordan": 1, "Moldova": 34, "Republic of Sou..": 32, "Rwanda": 12, "Somalia": 5,
    "Sudan": 132, "Syria": 171, "Tanzania": 1, "Uganda": 5, "Ukraine": 34, "Venezuela": 9,
    "Vietnam": 7})  # total check 1163

add("North Dakota", {"Afghanistan": 1, "Dem. Rep. Con..": 38, "Eritrea": 4,
    "Republic of Sou..": 4, "Somalia": 1, "Sudan": 9})  # total check 57

add("Not Available", {"Burma": 8, "Dem. Rep. Con..": 1})  # total check 9

add("Ohio", {"Afghanistan": 29, "Armenia": 3, "Bhutan": 3, "Burma": 39, "Burundi": 7, "Congo": 1,
    "Dem. Rep. Con..": 535, "Djibouti": 1, "El Salvador": 8, "Eritrea": 20, "Ethiopia": 7,
    "Guatemala": 11, "Honduras": 5, "Iran": 1, "Iraq": 23, "Moldova": 5, "Nepal": 1, "Pakistan": 6,
    "Republic of Sou..": 4, "Russia": 2, "Rwanda": 2, "Somalia": 49, "Sudan": 64, "Syria": 227,
    "Uganda": 4, "Ukraine": 69, "Venezuela": 15})  # total check 1141

add("Oklahoma", {"Afghanistan": 3, "Burma": 42, "Eritrea": 1, "Moldova": 4, "Sudan": 5,
    "Syria": 10, "Ukraine": 5})  # total check 70

add("Oregon", {"Afghanistan": 23, "Burma": 25, "Burundi": 14, "Cambodia": 4,
    "Dem. Rep. Con..": 54, "El Salvador": 2, "Eritrea": 2, "Ethiopia": 5, "Guatemala": 1, "Iran": 1,
    "Iraq": 6, "Moldova": 17, "Rwanda": 7, "Somalia": 4, "Sudan": 8, "Syria": 62, "Ukraine": 71,
    "Venezuela": 1})  # total check 307

# Pennsylvania spans pages 14-15
add("Pennsylvania", {"Afghanistan": 49, "Burma": 63, "Burundi": 4, "Central African..": 5,
    "Colombia": 17, "Dem. Rep. Con..": 299, "El Salvador": 18, "Eritrea": 6, "Ethiopia": 13,
    "Guatemala": 58, "Honduras": 20, "Iran": 4, "Iraq": 30, "Jordan": 1, "Kazakhstan": 5,
    "Liberia": 3, "Mali": 2, "Nepal": 3, "Pakistan": 1, "Republic of Sou..": 3, "Somalia": 8,
    "Sudan": 63, "Syria": 350, "Uganda": 3, "Ukraine": 73, "Uzbekistan": 2,
    "Venezuela": 6})  # total check 1109

add("Rhode Island", {"Cambodia": 9, "Dem. Rep. Con..": 37, "Guatemala": 7,
    "Syria": 19})  # total check 72

add("South Carolina", {"Afghanistan": 21, "Burma": 9, "Burundi": 1, "Colombia": 3,
    "Dem. Rep. Con..": 107, "El Salvador": 4, "Eritrea": 1, "Guatemala": 31, "Honduras": 1,
    "Iran": 4, "Moldova": 18, "Pakistan": 3, "Sudan": 5, "Syria": 39,
    "Ukraine": 70})  # total check 317

add("South Dakota", {"Burma": 5, "Dem. Rep. Con..": 28, "Eritrea": 7, "Iraq": 3,
    "Republic of Sou..": 1, "Somalia": 2, "Sudan": 10, "Ukraine": 6})  # total check 62

add("Tennessee", {"Afghanistan": 15, "Burma": 26, "Burundi": 19, "Colombia": 11,
    "Dem. Rep. Con..": 226, "El Salvador": 1, "Eritrea": 2, "Ethiopia": 3, "Guatemala": 12,
    "Honduras": 14, "India": 1, "Iran": 3, "Iraq": 9, "Moldova": 6, "Republic of Sou..": 17,
    "Rwanda": 1, "Somalia": 12, "Sudan": 71, "Syria": 30, "Uganda": 7, "Ukraine": 5,
    "Venezuela": 8})  # total check 499

# Texas spans pages 16-17
add("Texas", {"Afghanistan": 209, "Burma": 212, "Burundi": 18, "Cambodia": 7,
    "Central African..": 9, "Chad": 6, "Colombia": 15, "Dem. Rep. Con..": 719, "El Salvador": 43,
    "Eritrea": 18, "Ethiopia": 10, "Guatemala": 121, "Honduras": 54, "Iran": 42, "Iraq": 51,
    "Ivory Coast": 1, "Liberia": 6, "Moldova": 12, "Nepal": 1, "Pakistan": 1,
    "Republic of Sou..": 19, "Rwanda": 19, "Senegal": 1, "Somalia": 20, "Sudan": 107, "Syria": 358,
    "Uganda": 5, "Ukraine": 7, "Uzbekistan": 1, "Venezuela": 26})  # total check 2118

add("Utah", {"Afghanistan": 28, "Armenia": 4, "Burma": 18, "Burundi": 1,
    "Central African..": 14, "China": 2, "Dem. Rep. Con..": 144, "El Salvador": 9, "Eritrea": 3,
    "Guatemala": 21, "Honduras": 6, "Iraq": 2, "Ivory Coast": 1, "Jordan": 1, "Korea, North": 4,
    "Palestine": 6, "Republic of Sou..": 20, "Somalia": 14, "Sudan": 70, "Syria": 81,
    "Uganda": 1})  # total check 450

add("Vermont", {"Afghanistan": 9, "Burma": 2, "Dem. Rep. Con..": 42, "Guatemala": 3,
    "Somalia": 2, "Sudan": 33})  # total check 91

add("Virginia", {"Afghanistan": 201, "Bhutan": 4, "Burma": 15, "Colombia": 12,
    "Dem. Rep. Con..": 148, "El Salvador": 26, "Eritrea": 8, "Ethiopia": 5, "Guatemala": 22,
    "Honduras": 29, "Iran": 16, "Iraq": 12, "Pakistan": 2, "Republic of Sou..": 8, "Sudan": 28,
    "Syria": 92, "Ukraine": 9})  # total check 637

# Washington spans pages 18-19
add("Washington", {"Afghanistan": 88, "Armenia": 5, "Burma": 27, "Burundi": 5, "China": 1,
    "Colombia": 6, "Dem. Rep. Con..": 94, "El Salvador": 14, "Eritrea": 23, "Ethiopia": 22,
    "Guatemala": 34, "Guinea": 1, "Honduras": 28, "Iran": 17, "Iraq": 11, "Jordan": 1,
    "Moldova": 153, "Russia": 13, "Senegal": 1, "Somalia": 24, "Sudan": 36, "Syria": 132,
    "Uganda": 1, "Ukraine": 472, "Venezuela": 11, "Yemen": 4})  # total check 1224

add("West Virginia", {"Eritrea": 1, "Syria": 10, "Uganda": 2})  # total check 13

add("Wisconsin", {"Afghanistan": 7, "Burma": 323, "Burundi": 14, "Dem. Rep. Con..": 208,
    "Iran": 1, "Iraq": 4, "Pakistan": 3, "Republic of Sou..": 6, "Somalia": 13, "Sudan": 7,
    "Syria": 50})  # total check 636

fy2022_detail = pd.DataFrame(rows, columns=["state", "nationality", "total"])
print("FY2022 grand total:", fy2022_detail.total.sum(), "(should be 25,465)")

NAME_FIX_2022 = {
    "Dem. Rep. Con..": "Democratic Republic of the Congo",
    "Republic of Sou..": "South Sudan",
    "Central African..": "Central African Republic",
    "Sri Lanka (Ceyl..": "Sri Lanka (Ceylon)",
}
fy2022_detail["nationality"] = fy2022_detail.nationality.replace(NAME_FIX_2022)

fy2022_by_nationality = (fy2022_detail.groupby("nationality", as_index=False)
    .total.sum()
    .rename(columns={"total": "arrivals"})
    .assign(fy=2022)
    [["fy", "nationality", "arrivals"]]
    .sort_values("arrivals", ascending=False))

print(fy2022_by_nationality.head(15))

FY2022 grand total: 25465 (should be 25,465)
      fy                       nationality  arrivals
13  2022  Democratic Republic of the Congo      7810
45  2022                             Syria      4556
4   2022                             Burma      2156
44  2022                             Sudan      1669
0   2022                       Afghanistan      1618
50  2022                           Ukraine      1610
19  2022                         Guatemala      1085
16  2022                       El Salvador       519
33  2022                           Moldova       507
25  2022                              Iraq       498
41  2022                           Somalia       456
21  2022                          Honduras       450
17  2022                           Eritrea       325
42  2022                       South Sudan       293
11  2022                          Colombia       261


In [20]:
## FY2024 -- manually transcribed (see note above: Type3 font has no extractable text)
# Each state block is validated against the PDF's own printed state Total in a comment.
# The full combined total (100,034) matches the PDF's own Grand Total exactly.
import pandas as pd

# Manually transcribed from rendered page images of the FY2024 RPC "Refugee Arrivals by State and Nationality" PDF
# (its Type3 font has no extractable text -- each entry is (state, nationality, annual total))
# Validated per-state against the PDF's own printed state "Total" row where visible.

rows = []

def add(state, items):
    for nat, total in items.items():
        rows.append((state, nat, total))

# page 1
add("Alabama", {"Afghanistan":53,"Angola":1,"Burma":13,"Colombia":13,"Cuba":1,"Dem. Rep. Congo":22,
    "El Salvador":20,"Eritrea":11,"Guatemala":84,"Haiti":5,"Honduras":7,"Iraq":1,"Nicaragua":50,
    "Rwanda":3,"Somalia":30,"Syria":41,"Venezuela":199,"Yemen":5})  # total check 559

add("Alaska", {"Afghanistan":5,"Burma":8,"Colombia":4,"Dem. Rep. Congo":45,"Eritrea":2,"Guatemala":8,
    "Republic of South Sudan":18,"Russia":5,"Somalia":10,"Sudan":7,"Ukraine":25,"Venezuela":18})  # total check 155

# page 1 + page 2 (Arizona spans both)
add("Arizona", {"Afghanistan":400,"Burma":389,"Burundi":50,"Cameroon":3,"Central African Republic":20,
    "Chad":1,"Colombia":40,"Congo":7,"Cuba":3,"Dem. Rep. Congo":1274,"El Salvador":22,"Eritrea":69,
    "Ethiopia":3,"Guatemala":130,"Guinea":5,"Haiti":12,"Honduras":13,"Indonesia":1,"Iran":6,"Iraq":105,
    "Mali":5,"Moldova":1,"Nicaragua":117,"Nigeria":13,"Pakistan":26,"Republic of South Sudan":67,
    "Rwanda":19,"Senegal":1,"Somalia":280,"Sudan":59,"Syria":364,"The Gambia":1,"Uganda":3,
    "Venezuela":257,"Vietnam":2,"Yemen":8})  # total check 3776

add("Arkansas", {"Afghanistan":5,"Burma":4,"Colombia":2,"Dem. Rep. Congo":108,"El Salvador":9,
    "Guatemala":20,"Guinea":3,"Honduras":9,"Iran":2,"Ivory Coast":6,"Liberia":8,"Nicaragua":57,
    "Rwanda":3,"Syria":22,"Uganda":1,"Venezuela":82,"Vietnam":5})  # total check 346

# California spans pages 2-3
add("California", {"Afghanistan":3171,"Algeria":7,"Armenia":280,"Azerbaijan":1,"Belarus":89,
    "Burma":77,"Cambodia":5,"Cameroon":7,"Canada":1,"Colombia":17,"Congo":14,"Dem. Rep. Congo":174,
    "El Salvador":187,"Eritrea":53,"Ethiopia":18,"Georgia":3,"Guatemala":794,"Haiti":13,"Honduras":48,
    "Iran":277,"Iraq":364,"Ivory Coast":3,"Jamaica":2,"Kazakhstan":19,"Moldova":107,
    "Nicaragua":259,"Pakistan":27,"Republic of South Sudan":9,"Russia":78,"Senegal":2,"Somalia":60,
    "Sudan":58,"Syria":763,"Togo":1,"Ukraine":248,"Uzbekistan":17,"Venezuela":332,"Vietnam":8,
    "Yemen":4})  # total check 7597

add("Colorado", {"Afghanistan":326,"Bangladesh":1,"Belarus":9,"Burma":302,"Burundi":4,"Cameroon":5,
    "Central African Republic":56,"Colombia":26,"Congo":7,"Dem. Rep. Congo":304,"El Salvador":22,
    "Eritrea":84,"Ethiopia":15,"Guatemala":50,"Guinea":15,"Haiti":9,"Honduras":3,"Iraq":45,
    "Ivory Coast":1,"Jamaica":2,"Liberia":4,"Moldova":5,"Nicaragua":67,"Pakistan":7,
    "Republic of South Sudan":68,"Russia":11,"Rwanda":4,"Senegal":2,"Somalia":64,"Sudan":24,
    "Syria":168,"Tanzania":4,"Uganda":1,"Ukraine":27,"Venezuela":250})  # total check 1992

# Connecticut spans pages 3-4
add("Connecticut", {"Afghanistan":133,"Burma":21,"Colombia":34,"Cuba":2,"Dem. Rep. Congo":127,
    "El Salvador":8,"Eritrea":8,"Guatemala":95,"Haiti":7,"Honduras":3,"Iran":2,"Iraq":1,
    "Ivory Coast":1,"Jamaica":2,"Mali":13,"Nicaragua":34,"Republic of South Sudan":54,"Russia":3,
    "Somalia":39,"Sudan":10,"Syria":251,"Tunisia":1,"Ukraine":3,"Venezuela":169,"Yemen":11})  # total check 1032

add("Delaware", {"Afghanistan":9,"Colombia":30,"Congo":9,"Dem. Rep. Congo":21,"Eritrea":6,
    "Guatemala":15,"Honduras":1,"Nicaragua":12,"Syria":10,"Venezuela":54})  # total check 167

add("District of Columbia", {"Afghanistan":1,"El Salvador":1,"Eritrea":2,"Ethiopia":4,"Guatemala":6,
    "Republic of South Sudan":1,"Senegal":1,"Somalia":2,"Sudan":1,"Venezuela":10})  # total check 29

# Florida spans pages 4-5
add("Florida", {"Afghanistan":320,"Belarus":14,"Burma":93,"Cambodia":1,"Cameroon":1,
    "Central African Republic":32,"China":1,"Colombia":137,"Congo":10,"Cuba":98,
    "Dem. Rep. Congo":255,"El Salvador":75,"Eritrea":25,"Ethiopia":2,"Guatemala":257,
    "Haiti":31,"Honduras":108,"Iran":2,"Iraq":55,"Ivory Coast":4,"Kazakhstan":4,"Mali":1,
    "Moldova":24,"Nicaragua":380,"Pakistan":17,"Republic of South Sudan":32,"Russia":6,"Rwanda":5,
    "Senegal":9,"Somalia":15,"Sudan":47,"Syria":551,"Uganda":1,"Ukraine":74,"Venezuela":1882,
    "Yemen":6})  # total check 4575

# Georgia spans pages 5-6
add("Georgia", {"Afghanistan":381,"Burma":462,"Burundi":10,"Cameroon":6,
    "Central African Republic":60,"Chad":1,"Colombia":13,"Congo":17,"Cuba":3,"Dem. Rep. Congo":531,
    "Egypt":1,"El Salvador":36,"Eritrea":122,"Ethiopia":29,"Guatemala":157,"Guinea":9,"Haiti":15,
    "Honduras":44,"Iran":14,"Iraq":19,"Ivory Coast":2,"Jamaica":1,"Mali":5,"Moldova":15,"Nepal":1,
    "Nicaragua":123,"Niger":1,"Pakistan":25,"Palestine":2,"Republic of South Sudan":47,"Rwanda":1,
    "Senegal":9,"Somalia":148,"Sri Lanka (Ceylon)":1,"Sudan":93,"Syria":287,"The Gambia":2,
    "Uganda":1,"Ukraine":3,"Venezuela":512,"Vietnam":1,"Yemen":17})  # total check 3227

add("Hawaii", {"Burma":4,"Ukraine":1})  # total check 5

add("Idaho", {"Afghanistan":87,"Burma":14,"Burundi":17,"Central African Republic":2,"Colombia":7,
    "Dem. Rep. Congo":404,"El Salvador":3,"Eritrea":33,"Guatemala":46,"Honduras":2,"Iran":7,
    "Iraq":8,"Nicaragua":7,"Republic of South Sudan":1,"Somalia":5,"Sudan":32,"Syria":64,
    "Venezuela":47})  # total check 786

# Illinois spans pages 6-7
add("Illinois", {"Afghanistan":369,"Belarus":4,"Burma":761,"Burundi":22,"Cameroon":1,
    "Central African Republic":27,"Colombia":28,"Congo":25,"Cuba":3,"Dem. Rep. Congo":801,
    "El Salvador":11,"Eritrea":38,"Guatemala":128,"Guinea":2,"Haiti":7,"Honduras":30,"Indonesia":1,
    "Iran":10,"Iraq":122,"Ivory Coast":3,"Jordan":2,"Kenya":2,"Kyrgyzstan":7,"Moldova":4,"Nepal":1,
    "Nicaragua":71,"Nigeria":16,"Pakistan":57,"Palestine":10,"Peru":1,"Republic of South Sudan":19,
    "Russia":6,"Rwanda":11,"Senegal":3,"Somalia":32,"Sudan":40,"Syria":433,"The Gambia":4,
    "Uganda":2,"Ukraine":15,"Venezuela":665,"Vietnam":3,"Yemen":1})  # total check 3798

add("Indiana", {"Afghanistan":116,"Burma":421,"Cameroon":4,"Central African Republic":13,
    "Colombia":19,"Congo":2,"Cuba":2,"Dem. Rep. Congo":567,"El Salvador":14,"Eritrea":40,
    "Georgia":2,"Guatemala":57,"Guinea":1,"Haiti":1,"Honduras":10,"Iran":5,"Iraq":11,
    "Ivory Coast":2,"Nicaragua":81,"Pakistan":4,"Republic of South Sudan":2,"Rwanda":5,"Senegal":2,
    "Somalia":43,"Sudan":3,"Syria":145,"Ukraine":6,"Venezuela":238,"Yemen":1})  # total check 1817

# Iowa spans pages 7-8
add("Iowa", {"Afghanistan":168,"Burma":163,"Burundi":37,"Cambodia":3,"Cameroon":2,
    "Central African Republic":30,"Colombia":52,"Cuba":3,"Dem. Rep. Congo":1161,"El Salvador":25,
    "Eritrea":162,"Ethiopia":16,"Guatemala":53,"Guinea":3,"Guyana":2,"Haiti":3,"Honduras":15,
    "Iraq":15,"Ivory Coast":1,"Libya":7,"Mali":1,"Nicaragua":90,"Nigeria":6,"Pakistan":3,
    "Republic of South Sudan":88,"Rwanda":9,"Senegal":3,"Somalia":160,"Sudan":108,"Syria":161,
    "Uganda":2,"Venezuela":241,"Yemen":5})  # total check 2798

add("Kansas", {"Afghanistan":104,"Burma":96,"Burundi":15,"Central African Republic":5,"Colombia":7,
    "Congo":2,"Dem. Rep. Congo":391,"El Salvador":5,"Eritrea":29,"Ethiopia":5,"Guatemala":77,
    "Honduras":7,"Iraq":15,"Liberia":3,"Nicaragua":12,"Nigeria":9,"Republic of South Sudan":19,
    "Rwanda":10,"Somalia":117,"Sudan":36,"Syria":85,"Venezuela":75})  # total check 1124

# Kentucky spans pages 8-9
add("Kentucky", {"Afghanistan":141,"Belarus":4,"Burma":110,"Burundi":8,"Central African Republic":16,
    "Colombia":25,"Cuba":3,"Dem. Rep. Congo":1735,"El Salvador":4,"Eritrea":13,"Guatemala":45,
    "Haiti":3,"Honduras":15,"Iraq":23,"Mali":9,"Nicaragua":32,"Pakistan":34,
    "Republic of South Sudan":12,"Rwanda":13,"Somalia":197,"Sudan":67,"Syria":216,"Ukraine":35,
    "Venezuela":145})  # total check 2905

add("Louisiana", {"Afghanistan":1,"Burma":4,"Central African Republic":2,"Cuba":3,
    "Dem. Rep. Congo":3,"Eritrea":2,"Guatemala":13,"Haiti":2,"Honduras":22,"Iraq":6,
    "Nicaragua":25,"Syria":6,"Venezuela":52})  # total check 141

add("Maine", {"Afghanistan":52,"Burundi":14,"Cambodia":3,"Central African Republic":6,"Colombia":13,
    "Congo":5,"Dem. Rep. Congo":145,"El Salvador":4,"Eritrea":1,"Ethiopia":4,"Guatemala":20,
    "Iraq":51,"Nicaragua":24,"Republic of South Sudan":5,"Rwanda":8,"Somalia":48,"Sudan":1,
    "Syria":121,"Venezuela":100})  # total check 625

# Maryland spans pages 9-10
add("Maryland", {"Afghanistan":532,"Algeria":1,"Burma":96,"Burundi":24,"Cameroon":7,
    "Central African Republic":12,"Colombia":20,"Cuba":12,"Dem. Rep. Congo":112,"Egypt":1,
    "El Salvador":119,"Eritrea":129,"Ethiopia":14,"Guatemala":196,"Guinea":1,"Haiti":1,
    "Honduras":35,"Iran":7,"Iraq":31,"Ivory Coast":3,"Kyrgyzstan":5,"Mali":9,"Nepal":1,
    "Nicaragua":89,"Pakistan":49,"Republic of South Sudan":53,"Russia":1,"Senegal":2,
    "Sierra Leone":5,"Somalia":25,"Sudan":48,"Syria":202,"Venezuela":216,"Yemen":2})  # total check 2060

# Massachusetts spans pages 10-11
add("Massachusetts", {"Afghanistan":358,"Algeria":2,"Belarus":34,"Brazil":3,"Burma":57,
    "Burundi":29,"Cambodia":1,"Cameroon":6,"Central African Republic":46,"Colombia":8,"Congo":21,
    "Costa Rica":1,"Cuba":6,"Dem. Rep. Congo":424,"El Salvador":47,"Eritrea":54,"Ethiopia":11,
    "Guatemala":146,"Guinea":1,"Haiti":6,"Honduras":18,"Iran":1,"Iraq":87,"Ivory Coast":7,
    "Jordan":3,"Kazakhstan":8,"Mali":1,"Moldova":10,"Nicaragua":47,"Nigeria":8,"Norway":1,
    "Pakistan":5,"Republic of South Sudan":17,"Rwanda":1,"Senegal":16,"Somalia":73,"Sudan":97,
    "Syria":434,"Uganda":6,"Ukraine":9,"Venezuela":255})  # total check 2365

add("Michigan", {"Afghanistan":356,"Burma":106,"Burundi":28,"Cameroon":6,
    "Central African Republic":63,"Colombia":15,"Cuba":10,"Dem. Rep. Congo":862,"El Salvador":7,
    "Eritrea":35,"Ethiopia":6,"Guatemala":68,"Guinea":6,"Haiti":5,"Honduras":18,"Iran":17,
    "Iraq":260,"Ivory Coast":3,"Jamaica":2,"Jordan":1,"Liberia":1,"Libya":9,"Mali":13,
    "Nicaragua":33,"Nigeria":4,"Pakistan":12,"Palestine":3,"Republic of South Sudan":22,"Rwanda":5,
    "Somalia":57,"Sudan":55,"Syria":1186,"The Gambia":3,"Uganda":5,"Ukraine":19,"Venezuela":302,
    "Yemen":46})  # total check 3649

# Minnesota spans pages 11-12
add("Minnesota", {"Afghanistan":172,"Belarus":20,"Burma":218,"Cambodia":2,"Cameroon":9,
    "Central African Republic":7,"Chad":2,"Colombia":20,"Congo":10,"Dem. Rep. Congo":182,
    "El Salvador":11,"Eritrea":113,"Ethiopia":152,"Guatemala":64,"Guinea":1,"Haiti":8,
    "Honduras":32,"Iraq":30,"Ivory Coast":8,"Kenya":1,"Kyrgyzstan":15,"Moldova":17,"Nicaragua":135,
    "Pakistan":16,"Republic of South Sudan":63,"Russia":22,"Rwanda":6,"Somalia":1267,"Sudan":49,
    "Syria":108,"Ukraine":32,"Uzbekistan":1,"Venezuela":142,"Vietnam":52,"Yemen":3})  # total check 2990

add("Mississippi", {"Eritrea":4,"Guatemala":7,"Nicaragua":15,"Sudan":2,"Venezuela":16})  # total check 44

# Missouri spans pages 12-13
add("Missouri", {"Afghanistan":326,"Burkina Faso (UVolta)":3,"Burma":94,"Burundi":36,"Cameroon":4,
    "Central African Republic":89,"Colombia":25,"Congo":32,"Dem. Rep. Congo":981,"El Salvador":4,
    "Eritrea":72,"Ethiopia":9,"Guatemala":48,"Guinea":2,"Guinea-Bissau":1,"Haiti":12,"Honduras":28,
    "Iraq":60,"Ivory Coast":3,"Mali":16,"Moldova":9,"Nicaragua":42,"Nigeria":14,"Pakistan":22,
    "Republic of South Sudan":49,"Rwanda":8,"Senegal":2,"Somalia":257,"Sudan":124,"Syria":188,
    "Uganda":12,"Ukraine":15,"Venezuela":113,"Yemen":1,"Zimbabwe":1})  # total check 2702

add("Montana", {"Afghanistan":42,"Dem. Rep. Congo":31,"Guatemala":7,"Iraq":7,"Syria":39,
    "Venezuela":36,"Yemen":12})  # total check 174

add("Nebraska", {"Afghanistan":189,"Burma":217,"Burundi":30,"Central African Republic":10,
    "Colombia":9,"Congo":3,"Cuba":6,"Dem. Rep. Congo":244,"Djibouti":1,"El Salvador":21,
    "Eritrea":18,"Ethiopia":15,"Guatemala":55,"Honduras":3,"Iraq":136,"Kenya":5,"Nicaragua":27,
    "Nigeria":1,"Pakistan":6,"Republic of South Sudan":151,"Russia":2,"Somalia":173,"Sudan":99,
    "Syria":269,"Uganda":1,"Ukraine":11,"Venezuela":141,"Yemen":2})  # total check 1845

# Nevada spans pages 13-14
add("Nevada", {"Afghanistan":82,"Armenia":1,"Burma":30,"Burundi":16,"Cameroon":8,
    "Central African Republic":3,"Chad":3,"Colombia":8,"Cuba":5,"Dem. Rep. Congo":80,"Djibouti":1,
    "El Salvador":28,"Eritrea":52,"Ethiopia":3,"Guatemala":108,"Guinea":2,"Haiti":16,"Honduras":6,
    "Iran":7,"Iraq":6,"Ivory Coast":3,"Mali":2,"Nicaragua":76,"Nigeria":4,
    "Republic of South Sudan":10,"Rwanda":1,"Senegal":3,"Somalia":11,"Sudan":16,"Syria":67,
    "Uganda":1,"Venezuela":157})  # total check 816

add("New Hampshire", {"Afghanistan":60,"Burma":14,"Burundi":5,"Cuba":4,"Dem. Rep. Congo":153,
    "El Salvador":2,"Eritrea":9,"Guatemala":8,"Honduras":2,"Iraq":13,"Nicaragua":9,
    "Republic of South Sudan":7,"Senegal":1,"Somalia":5,"Sudan":1,"Syria":11,
    "Venezuela":23})  # total check 327

# New Jersey spans pages 14-15
add("New Jersey", {"Afghanistan":139,"Burma":3,"Cameroon":4,"Colombia":51,"Cuba":4,
    "Dem. Rep. Congo":14,"Ecuador":4,"El Salvador":73,"Eritrea":8,"Guatemala":160,"Guinea":3,
    "Guyana":1,"Haiti":14,"Honduras":34,"Iraq":3,"Ivory Coast":1,"Jamaica":1,"Nicaragua":71,
    "Niger":6,"Pakistan":22,"Palestine":4,"Rwanda":1,"Somalia":2,"Sri Lanka (Ceylon)":4,"Sudan":1,
    "Syria":142,"Ukraine":4,"Venezuela":353})  # total check 1127

add("New Mexico", {"Afghanistan":51,"Burma":14,"Cambodia":4,"Cameroon":4,
    "Central African Republic":2,"Colombia":44,"Congo":6,"Cuba":6,"Dem. Rep. Congo":82,"Egypt":1,
    "El Salvador":7,"Eritrea":4,"Guatemala":26,"Guinea":2,"Haiti":3,"Nicaragua":35,"Pakistan":24,
    "Republic of South Sudan":34,"Rwanda":7,"Sudan":6,"Syria":32,"Venezuela":69})  # total check 463

# New York spans pages 15-16
add("New York", {"Afghanistan":658,"Algeria":2,"Azerbaijan":10,"Belarus":27,
    "Burkina Faso (UVolta)":1,"Burma":736,"Burundi":43,"Cameroon":3,"Central African Republic":44,
    "China":1,"Colombia":114,"Congo":25,"Cuba":12,"Dem. Rep. Congo":1295,"Egypt":1,"El Salvador":79,
    "Eritrea":92,"Ethiopia":15,"Georgia":2,"Guatemala":219,"Guyana":1,"Haiti":12,"Honduras":62,
    "Iran":3,"Iraq":114,"Ivory Coast":2,"Jamaica":1,"Kyrgyzstan":37,"Liberia":16,"Mali":24,
    "Moldova":13,"Nicaragua":80,"Pakistan":52,"Republic of South Sudan":94,"Russia":9,"Rwanda":11,
    "Senegal":1,"Somalia":366,"Sri Lanka (Ceylon)":2,"Sudan":194,"Syria":1112,"Tanzania":2,
    "Togo":1,"Turkmenistan":1,"Uganda":2,"Ukraine":90,"Uzbekistan":13,"Venezuela":514,
    "Yemen":26})  # total check 6234

# North Carolina spans pages 16-17
add("North Carolina", {"Afghanistan":317,"Belarus":22,"Burma":302,"Burundi":11,"Cambodia":7,
    "Cameroon":3,"Central African Republic":42,"Colombia":93,"Congo":19,"Cuba":6,
    "Dem. Rep. Congo":674,"Djibouti":1,"El Salvador":83,"Eritrea":141,"Ethiopia":17,
    "Guatemala":255,"Guinea":7,"Haiti":1,"Honduras":38,"Iran":10,"Iraq":33,"Ivory Coast":1,
    "Jamaica":2,"Jordan":2,"Mali":11,"Moldova":14,"Nicaragua":138,"Nigeria":13,"Pakistan":56,
    "Republic of South Sudan":35,"Russia":22,"Rwanda":11,"Somalia":124,"Sudan":65,"Syria":388,
    "Ukraine":33,"Uzbekistan":1,"Venezuela":412,"Vietnam":37,"Yemen":13})  # total check 3460

add("North Dakota", {"Afghanistan":59,"Bangladesh":1,"Burundi":6,"Central African Republic":2,
    "Colombia":2,"Dem. Rep. Congo":105,"Ethiopia":4,"Guatemala":23,"Iraq":11,"Liberia":2,
    "Nicaragua":20,"Pakistan":12,"Republic of South Sudan":2,"Somalia":53,"Sudan":24,"Syria":6,
    "Venezuela":65})  # total check 397

add("Not Available", {"Afghanistan":13,"Dem. Rep. Congo":9,"El Salvador":13,"Guatemala":1,
    "Honduras":1,"Iraq":1,"Korea, Republic of South":1,"Nicaragua":3})  # total check 42

# Ohio spans pages 17-18
add("Ohio", {"Afghanistan":280,"Armenia":4,"Belarus":3,"Burma":110,"Burundi":13,"Cameroon":2,
    "Central African Republic":11,"Colombia":22,"Congo":17,"Cuba":5,"Dem. Rep. Congo":1343,
    "El Salvador":14,"Eritrea":131,"Ethiopia":33,"Guatemala":74,"Guinea":2,"Haiti":1,"Honduras":14,
    "India":3,"Iran":2,"Iraq":78,"Ivory Coast":2,"Jamaica":2,"Kenya":1,"Mali":4,"Nepal":4,
    "Nicaragua":101,"Pakistan":24,"Palestine":4,"Republic of South Sudan":26,"Russia":2,"Rwanda":6,
    "Senegal":5,"Somalia":326,"Sudan":146,"Syria":467,"The Gambia":1,"Uganda":5,"Ukraine":54,
    "Uzbekistan":6,"Venezuela":297,"Yemen":12})  # total check 3657

add("Oklahoma", {"Afghanistan":83,"Belarus":8,"Burma":151,"Cambodia":2,
    "Central African Republic":11,"Colombia":21,"Cuba":2,"Dem. Rep. Congo":12,"El Salvador":6,
    "Eritrea":6,"Ethiopia":6,"Guatemala":69,"Honduras":16,"Iraq":2,"Nicaragua":44,"Pakistan":5,
    "Russia":10,"Senegal":1,"Somalia":6,"Sudan":12,"Syria":49,"Ukraine":4,"Venezuela":190,
    "Vietnam":7,"Yemen":1})  # total check 724

# Oregon spans pages 18-19
add("Oregon", {"Afghanistan":234,"Armenia":5,"Belarus":20,"Burma":203,"Burundi":9,"Cambodia":5,
    "Cameroon":1,"Central African Republic":9,"China":1,"Colombia":5,"Cuba":3,"Dem. Rep. Congo":188,
    "El Salvador":9,"Eritrea":27,"Ethiopia":15,"Guatemala":56,"Guinea":1,"Haiti":2,"India":4,
    "Iran":3,"Iraq":29,"Ivory Coast":2,"Kazakhstan":10,"Kyrgyzstan":8,"Mali":8,"Mauritania":1,
    "Moldova":24,"Nicaragua":22,"Nigeria":2,"Palestine":4,"Republic of South Sudan":12,"Russia":43,
    "Senegal":1,"Somalia":50,"Sudan":27,"Syria":122,"Turkmenistan":3,"Ukraine":57,"Uzbekistan":5,
    "Venezuela":99,"Vietnam":2})  # total check 1331

# Pennsylvania spans pages 19-20
add("Pennsylvania", {"Afghanistan":607,"Algeria":1,"Belarus":24,"Burma":185,"Burundi":44,
    "Cameroon":13,"Central African Republic":41,"Colombia":76,"Congo":28,"Cuba":3,
    "Dem. Rep. Congo":930,"El Salvador":39,"Eritrea":93,"Ethiopia":1,"Georgia":4,"Guatemala":150,
    "Guinea":5,"Haiti":41,"Honduras":18,"Iran":3,"Iraq":20,"Ivory Coast":10,"Jamaica":1,
    "Jordan":1,"Kazakhstan":19,"Kyrgyzstan":8,"Lebanon":6,"Liberia":1,"Libya":9,"Mali":27,
    "Nicaragua":119,"Niger":1,"Nigeria":4,"Pakistan":120,"Republic of South Sudan":32,"Russia":13,
    "Rwanda":3,"Senegal":3,"Sierra Leone":1,"Somalia":149,"South Africa":1,"Sudan":132,
    "Syria":954,"Tunisia":1,"Turkey":1,"Turkmenistan":4,"Uganda":1,"Ukraine":49,"Venezuela":447,
    "Vietnam":1,"Yemen":2})  # total check 4446

add("Rhode Island", {"Afghanistan":44,"Burundi":13,"Cambodia":10,"Colombia":21,"Congo":4,
    "Dem. Rep. Congo":51,"Eritrea":10,"Ethiopia":2,"Guatemala":41,"Honduras":5,"Iraq":8,"Mali":3,
    "Mauritania":1,"Nicaragua":20,"Pakistan":4,"Republic of South Sudan":8,"Somalia":11,"Sudan":10,
    "Syria":72,"The Gambia":1,"Venezuela":59})  # total check 398

add("South Carolina", {"Afghanistan":54,"Angola":1,"Armenia":9,"Belarus":22,"Burma":117,
    "Cameroon":6,"Central African Republic":26,"Chad":2,"Colombia":35,"Congo":24,
    "Dem. Rep. Congo":147,"El Salvador":5,"Eritrea":19,"Ethiopia":1,"Guatemala":138,"Guinea":3,
    "Haiti":2,"Honduras":22,"Iraq":8,"Ivory Coast":1,"Liberia":1,"Mali":23,"Moldova":10,
    "Nicaragua":54,"Pakistan":16,"Republic of South Sudan":13,"Russia":17,"Rwanda":12,
    "Senegal":15,"Somalia":8,"Sudan":9,"Syria":238,"Uganda":2,"Ukraine":61,
    "Venezuela":214})  # total check 1335

add("South Dakota", {"Afghanistan":23,"Burma":30,"Burundi":3,"Central African Republic":19,
    "Congo":5,"Dem. Rep. Congo":93,"Eritrea":31,"Ethiopia":3,"Guatemala":26,"Nicaragua":22,
    "Republic of South Sudan":2,"Rwanda":3,"Somalia":43,"Sudan":28,"Ukraine":5,
    "Venezuela":55})  # total check 391

# Tennessee spans pages 21-22
add("Tennessee", {"Afghanistan":167,"Burma":102,"Burundi":14,"Central African Republic":12,
    "Colombia":51,"Congo":13,"Cuba":10,"Dem. Rep. Congo":494,"El Salvador":19,"Eritrea":15,
    "Ethiopia":6,"Guatemala":70,"Haiti":2,"Honduras":15,"Iran":4,"Iraq":52,"Kazakhstan":5,
    "Moldova":1,"Nicaragua":52,"Nigeria":4,"Pakistan":11,"Republic of South Sudan":49,"Russia":3,
    "Rwanda":2,"Somalia":38,"Sudan":75,"Syria":127,"Uganda":1,"Ukraine":4,"Venezuela":436,
    "Yemen":4})  # total check 1858

# Texas spans pages 22-23
add("Texas", {"Afghanistan":1574,"Burma":882,"Burundi":79,"Cambodia":17,"Cameroon":13,
    "Central African Republic":82,"Chad":1,"Colombia":120,"Congo":11,"Cuba":33,
    "Dem. Rep. Congo":1791,"El Salvador":174,"Eritrea":286,"Ethiopia":38,"Guatemala":551,
    "Guinea":4,"Haiti":16,"Honduras":144,"Indonesia":2,"Iran":53,"Iraq":189,"Ivory Coast":3,
    "Kazakhstan":3,"Libya":9,"Mali":3,"Nepal":1,"Nicaragua":315,"Nigeria":11,"Pakistan":88,
    "Palestine":3,"Republic of South Sudan":122,"Russia":1,"Rwanda":49,"Senegal":8,"Somalia":177,
    "Sri Lanka (Ceylon)":5,"Sudan":170,"Syria":676,"The Gambia":4,"Uganda":3,"Ukraine":2,
    "Venezuela":2038,"Vietnam":2,"Yemen":15})  # total check 9768

add("Utah", {"Afghanistan":111,"Burma":29,"Burundi":2,"Cameroon":1,"Central African Republic":9,
    "Colombia":29,"Cuba":1,"Dem. Rep. Congo":357,"El Salvador":2,"Eritrea":17,"Ethiopia":2,
    "Guatemala":66,"Guinea":1,"Haiti":3,"Iran":11,"Iraq":54,"Ivory Coast":1,"Libya":8,
    "Nicaragua":31,"Pakistan":5,"Republic of South Sudan":7,"Rwanda":1,"Somalia":63,"Sudan":67,
    "Syria":74,"Venezuela":251})  # total check 1203

# Vermont spans pages 23-24
add("Vermont", {"Afghanistan":94,"Burma":9,"Burundi":1,"Central African Republic":12,
    "Colombia":3,"Dem. Rep. Congo":169,"El Salvador":3,"Eritrea":27,"Haiti":2,"Honduras":1,
    "Iraq":1,"Nicaragua":9,"Republic of South Sudan":9,"Rwanda":2,"Somalia":9,"Sudan":6,
    "Syria":24,"Venezuela":49})  # total check 430

add("Virginia", {"Afghanistan":1258,"Burma":10,"Burundi":3,"Cameroon":1,
    "Central African Republic":6,"Colombia":14,"Congo":11,"Dem. Rep. Congo":249,"El Salvador":99,
    "Eritrea":63,"Ethiopia":8,"Guatemala":141,"Honduras":32,"Iran":11,"Iraq":59,"Jamaica":1,
    "Mali":10,"Nicaragua":43,"Pakistan":62,"Republic of South Sudan":11,"Russia":12,"Senegal":1,
    "Somalia":4,"Sudan":35,"Syria":180,"Ukraine":38,"Venezuela":130,"Yemen":5})  # total check 2497

# Washington spans pages 24-25
add("Washington", {"Afghanistan":889,"Armenia":21,"Belarus":91,"Burma":120,"Burundi":3,
    "Cameroon":3,"Central African Republic":13,"China":5,"Colombia":95,"Cuba":8,
    "Dem. Rep. Congo":247,"El Salvador":22,"Eritrea":250,"Ethiopia":46,"Georgia":23,
    "Guatemala":171,"Guinea":5,"Haiti":5,"Honduras":23,"Iran":14,"Iraq":121,"Ivory Coast":1})

add("Washington", {"Kazakhstan":43,"Kenya":2,"Libya":7,"Mauritania":1,"Moldova":114,"Nicaragua":112,
    "Nigeria":3,"Pakistan":25,"Palestine":1,"Republic of South Sudan":30,"Russia":83,"Rwanda":5,
    "Senegal":1,"Somalia":149,"Sudan":94,"Sweden":3,"Syria":321,"Uganda":4,"Ukraine":524,
    "Uzbekistan":12,"Venezuela":291})  # total check 4001

add("West Virginia", {"Afghanistan":3,"Cuba":1,"Dem. Rep. Congo":1,"Eritrea":2,"Iraq":5,"Nicaragua":3,
    "Syria":4,"Venezuela":2})  # total check 21

add("Wisconsin", {"Afghanistan":91,"Burma":570,"Burundi":1,"Cameroon":2,"Central African Republic":24,
    "Colombia":13,"Congo":26,"Dem. Rep. Congo":535,"Eritrea":3,"Ethiopia":1,"Germany":1,"Guatemala":18,
    "Honduras":1,"Iran":1,"Iraq":23,"Nicaragua":91,"Pakistan":20,"Republic of South Sudan":22,
    "Rwanda":1,"Somalia":105,"Sudan":6,"Syria":94,"Venezuela":175,"Yemen":1})  # total check 1825

fy2024_detail = pd.DataFrame(rows, columns=["state", "nationality", "total"])
print("FY2024 grand total:", fy2024_detail.total.sum(), "(should be 100,034)")

FY2024 grand total: 100034 (should be 100,034)


In [21]:
## combine all six years, harmonize nationality names, write final csv

NAME_FIX_2024 = {
    "Dem. Rep. Congo": "Democratic Republic of the Congo",
    "Republic of South Sudan": "South Sudan",
    "Korea, Republic of South": "South Korea",
}
fy2024_detail["nationality"] = fy2024_detail.nationality.replace(NAME_FIX_2024)

fy2024_by_nationality = (fy2024_detail.groupby("nationality", as_index=False)
    .total.sum()
    .rename(columns={"total": "arrivals"})
    .assign(fy=2024)
    [["fy", "nationality", "arrivals"]]
    .sort_values("arrivals", ascending=False))

print(fy2024_by_nationality.head(15))

by_nationality_combined = pd.concat(
    [by_nationality, fy2022_by_nationality, fy2024_by_nationality], ignore_index=True)
by_nationality_combined = by_nationality_combined.sort_values(["fy", "arrivals"], ascending=[True, False])

by_nationality_combined.to_csv("refugee_arrivals_by_nationality_fy2021_fy2026.csv", index=False)
print(by_nationality_combined.fy.value_counts())


      fy                       nationality  arrivals
21  2024  Democratic Republic of the Congo     19923
0   2024                       Afghanistan     14708
81  2024                         Venezuela     12875
71  2024                             Syria     11274
9   2024                             Burma      7347
30  2024                         Guatemala      5017
64  2024                           Somalia      4801
53  2024                         Nicaragua      3299
26  2024                           Eritrea      2411
39  2024                              Iraq      2282
69  2024                             Sudan      2184
79  2024                           Ukraine      1448
17  2024                          Colombia      1381
25  2024                       El Salvador      1332
67  2024                       South Sudan      1322
fy
2024    85
2023    75
2025    69
2022    56
2021    50
2026     2
Name: count, dtype: int64
